In [4]:
import re
from urllib.parse import urlparse
import pandas as pd
import math
from collections import Counter
from urllib.parse import urlparse

#### Address Bar Based Features

In [5]:
def url_length(url):
    return len(url)

def has_at(url):
    if '@' in url:
        return 1
    return 0

def has_dash(url):
    return url.count('-')

def dot(url):
    return url.count('.')

def no_of_subdomains(url):
    hostname = urlparse(url).netloc
    return hostname.count('.')

def has_ip(url):
    parsed = urlparse(url)
    hostname = parsed.netloc
    if re.search(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}',hostname):
        return 1
    return 0

def shortening_service(url):
    shortners = pd.read_csv('../data/datasets'
    '/shortners.csv')
    return 1 if any(s in url for s in shortners) else 0

def double_slash_redirect(url):
    return url.count('//') if url.count('//') > 1 else 0

def ssl_final_state(url):
    return int(not url.startswith("https://"))

def https_token(url):
    hostname = urlparse(url).netloc.lower()
    return int("https" in hostname)

def redirect_symbol(url):
    return int('//' in url[8:])

def suspicious_words(url):
    keywords = ['login', 'verify', 'update', 'secure', 'account', 'banking', 'support', 'password', 'confirm']
    url_lower = url.lower()
    return sum(url_lower.count(word) for word in keywords)

def count_digits(url):
    return sum(c.isdigit() for c in url)

def path_length(url):
    parsed = urlparse(url)
    return len(parsed.path)

def suspicious_tld(url):
    bad_tlds = ['.xyz', '.top', '.tk', '.ml', '.ga', '.cf', '.gq', '.club', '.site', '.online', '.cc', '.su']
    parsed_domain = urlparse(url).netloc.lower()
    if ':' in parsed_domain:
        parsed_domain = parsed_domain.split(':')[0]
    return 1 if any(parsed_domain.endswith(tld) for tld in bad_tlds) else 0

def calculate_entropy(url):
    entropy = 0
    for x in Counter(url).values():
        px = float(x)/len(url)
        entropy -= px * math.log(px,2)
    return entropy

def domain_length(url):
    return len(urlparse(url).netloc)

def count_paramaters(url):
    query = urlparse(url).query
    return len(query.split('&')) if query else 0

def has_punycode(url):
    return 1 if 'xn--' in url.lower() else 0

def count_special_chars(url):
    special_chars = ['?', '=', '_', '~', '%']
    return sum(url.count(c) for c in special_chars)

def extract_all_features(url):
    return pd.Series({
        'url_length' : url_length(url),
        'has_at' : has_at(url),
        'has_dash' : has_dash(url),
        'dot' : dot(url),
        'has_ip': has_ip(url),
        'shortening_service' : shortening_service(url),
        'double_slash_redirect' : double_slash_redirect(url),
        'ssl_final_state' : ssl_final_state(url),
        'https_token' : https_token(url),
        'redirect_symbol' : redirect_symbol(url),
        'suspicious_words' : suspicious_words(url),
        'digits' : count_digits(url),
        'path_length' : path_length(url),
        'suspicious_tld' : suspicious_tld(url),
        'entropy' : calculate_entropy(url),
        'domain_length' : domain_length(url),
        'paramaters' : count_paramaters(url),
        'has_punycode' : has_punycode(url),
        'special_chars' : count_special_chars(url)
    })

In [6]:
data = pd.read_csv('../data/datasets/raw_dataset.csv')
columns = data['url'].apply(extract_all_features)
columns['label'] = data['label']

columns.to_csv('../data/datasets/processed_dataset_v2.csv',index=False)
print("Raw data is processed successfully...")

Raw data is processed successfully...
